In [ ]:
#| default_exp callbacks.legacy

# callbacks.legacy

> **Quarantine ward.** Pre-Soft-era CBs kept for backward compatibility only.
> Do NOT use in new handlers — migrate to the `Soft*` equivalents in `core.py`.
> ⚠ `RemoveAllNAValuesCB` contains `dropna()` — Fail-Fast violation, grandfathered.
> Auto-generated by nbdev from `nbs/api/callbacks/legacy.ipynb`.

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
from typing import Dict, Union
from fastcore.all import store_attr
from marisco.callbacks.core import Callback, PerGroupCB, Transformer
from marisco.configs import SMP_TYPE_LUT


## Schema alignment (legacy)

In [ ]:
#| export
class AddSampleTypeIdColumnCB(PerGroupCB):
    "Add a column with the sample type as defined in the CDL."
    def __init__(self, 
                 lut: dict=SMP_TYPE_LUT, # Lookup table for sample type
                 col_name: str='SAMPLE_TYPE' # Column name to store the sample type id
                 ): 
        store_attr()
        
    def each_grp(self, grp, df, tfm): df[self.col_name] = self.lut[grp]

In [ ]:
#| export
class RenameColumnsCB(PerGroupCB):
    "Rename variables to MARIS standard names, keeping only renamed columns."
    def __init__(self,
                 renaming_rules: dict # Renaming rules {old_name: new_name}
                 ): 
        store_attr()
        
    def each_grp(self, grp, df, tfm): tfm.dfs[grp] = df[self.renaming_rules.keys()].rename(columns=self.renaming_rules)


#| export
## Row removal (legacy)

In [ ]:
#| export
class RemoveAllNAValuesCB(Callback):
    "Remove rows with all NA values in specified columns."
    def __init__(self, 
                 cols_to_check: Union[Dict[str, list], list],  # Dict or list of columns to check
                 how: str='all'  # How to handle NA values 'all' or 'any'
                ):
        store_attr()

    def __call__(self, tfm):
        # Convert list to dict if cols_to_check is a list
        cols_dict = (self.cols_to_check if isinstance(self.cols_to_check, dict) 
                    else {k: self.cols_to_check for k in tfm.dfs.keys()})
        
        for sample_type, columns in cols_dict.items():
            tfm.dfs[sample_type].dropna(  # grandfathered
                subset=columns,
                how=self.how,
                inplace=True
            )

In [ ]:
#| export
class MeltWideNuclidesCB(Callback):
    "Reshape wide nuclide columns to long format using a named-dict spec."
    def __init__(self,
                 spec: list,           # List of dicts with keys: val, unc, nuclide, unit, lab
                 grp:  str='SEAWATER', # Group in tfm.dfs to reshape
                 ):
        store_attr()

    def __call__(self, tfm):
        if self.grp not in tfm.dfs: return
        df = tfm.dfs[self.grp]
        frames = []
        for s in self.spec:
            sub = df.dropna(subset=[s['val']]).copy()  # grandfathered
            sub['NUCLIDE'] = s['nuclide']
            sub['VALUE']   = sub[s['val']]
            sub['UNC']     = sub[s['unc']]
            sub['UNIT']    = s['unit']
            sub['LAB']     = s['lab']
            frames.append(sub)
        if frames:
            tfm.dfs[self.grp] = pd.concat(frames, ignore_index=True)

#| export
## Comparison & audit

In [ ]:
#| export
class CompareDfsAndTfmCB(Callback):
    "Create a dataframe of removed data and track changes in row counts due to transformations."  # TODO: refactor - too long
    def __init__(self, 
                 dfs: Dict[str, pd.DataFrame]  # Original dataframes
                 ): 
        store_attr()
        
    def __call__(self, tfm: Transformer) -> None:
        self._initialize_tfm_attributes(tfm)
        for grp in tfm.dfs.keys():
            self._compute_changes(grp, tfm)

    def _initialize_tfm_attributes(self, tfm: Transformer) -> None:
        tfm.dfs_removed = {}
        tfm.compare_stats = {}

    def _compute_changes(self, 
                         grp: str,  # The group key
                         tfm: Transformer  # The transformation object containing `dfs`
                        ) -> None:
        "Compute and store changes including data removed and created during transformation."
        original_df = self.dfs[grp]
        transformed_df = tfm.dfs[grp]

        # Calculate differences
        original_count = len(original_df.index)
        transformed_count = len(transformed_df.index)
        removed_count = len(original_df.index.difference(transformed_df.index))
        created_count = len(transformed_df.index.difference(original_df.index))

        # Store results
        tfm.dfs_removed[grp] = original_df.loc[original_df.index.difference(transformed_df.index)]
        tfm.compare_stats[grp] = {
            'Original row count (dfs)': original_count,
            'Transformed row count (tfm.dfs)': transformed_count,
            'Rows removed from original (tfm.dfs_removed)': removed_count,
            'Rows created in transformed (tfm.dfs_created)': created_count
        }

In [ ]:
#| export
class UniqueIndexCB(PerGroupCB):
    "Set unique index for each group."
    def __init__(self, index_name='ID'): store_attr()
        
    def each_grp(self, grp, df, tfm):
        tfm.dfs[grp] = df.reset_index(drop=True).reset_index(names=[self.index_name])

In [ ]:
#| export
class ParseTimeCB(PerGroupCB):
    "Parse time column from ISO8601 string to datetime."
    def __init__(self, time_col_name: str='TIME'): store_attr()
    def each_grp(self, grp, df, tfm):
        df[self.time_col_name] = pd.to_datetime(df[self.time_col_name], format='ISO8601')

---
> **Migration guide**: Replace `MeltWideNuclidesCB` → `SoftMeltWideNuclidesCB`,
> `ParseTimeCB` → `SoftParseDateTimeCB`, `RenameColumnsCB` → `RenameColsCB`,
> `RemoveAllNAValuesCB` → filter before passing to Transformer.
